I. Зображення як багатовимірні масиви та підготовка зображень до подальшого аналізу
===================================================================================

_Image Analysis with Python and Napari, Bioinformatics for Ukraine course, 6-24 October 2025, Kyiv, Ukraine._

_© Borys Olifirov, 2025_

__План:__
- Основні властивості та відмінності списків _python_ та масивів _numpy_
- Багатовимірні масиви _numpy_, індексація та операції з багатовимірними масивами
- Зчитування та відображення зображень, кольорові мапи
- Попередня обробка зображень, компенсація фонової інтенсивності
- Функції python (`def`)

---

In [ ]:
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt

import skimage  # scikit-image

# Списки vs. масиви
---

### Основні властивості

Списки python

In [ ]:
demo_list = [0, 1, 4, 6, 2]
demo_list

In [ ]:
try:
    demo_list > 1
except TypeError:
    print('Imposible operation!')

In [ ]:
demo_list * 2  # подвоєння списку

In [ ]:
demo_list = [1, False, 3, 'a']  # список може містити різні типи даних
print(demo_list)

print(type(demo_list[0]))
print(type(demo_list[1]))
print(type(demo_list[-1]))

Масиви numpy

In [ ]:
demo_arr = np.array([0, 1, 4, 6, 2])
print(demo_arr)
print(demo_arr.dtype)  # всі елементи масиву відносяться до одного типу даних

In [ ]:
demo_arr_mixed = np.array([0, 1, 4, 'a', 2])
print(demo_arr_mixed)
print(demo_arr_mixed.dtype)  # Unicode, символьний тип дних

In [ ]:
demo_arr > 1

In [ ]:
demo_arr_mixed > 1

In [ ]:
demo_arr * 2  # застосування операції до кожного елементу масиву, векторизація

In [ ]:
demo_arr_mixed * 2

### Багатовимірність та індексація

Двовимірний список

In [ ]:
list_2d = [[1,2],[3,4],[5,6]]
list_2d

Двовимірний масив та їх індексація

In [ ]:
arr_2d = np.array([[1,1,1],[2,2,2],[3,3,3]])  # "зображення" розміром 3x3 пікседі

print(arr_2d.shape)
arr_2d

In [ ]:
arr_2d[0,0]

In [ ]:
arr_2d[:2,0]

In [ ]:
arr_2d[-1,-1]

In [ ]:
arr_2d[0:2]  # другий індекс не включається в зріз

In [ ]:
arr_2d[1:]  # перший індекс включається в зріз

Векторизовані операції з масивами (vectorization)

In [ ]:
arr_2d * 2

In [ ]:
arr_2d[:2] >= 3

Трансляція операцій з масивами (broadcasting)

In [ ]:
arr_2d + np.array([1,1])  # векторизовані операції із двома масивами можливі, якщо співпадають розмірності

In [ ]:
arr_2d + np.array([1,2,3])  # додавання вектора по рядках

In [ ]:
arr_2d + np.array([[1],[2],[3]])  # додавання вектора по стовбцях

Тривимірні масиви та їх індексація

In [ ]:
arr_3d = np.array([[[0,0],  # 3 "кадри" розміром 2x2 пікселі
                    [0,0]],
                   [[1,1],
                    [1,1]],
                   [[2,2],
                    [2,2]]])

print(arr_3d.shape)
arr_3d

In [ ]:
arr_3d[0,0,0]

In [ ]:
for frame in arr_3d:  # ітерація відбувається по першому виміру масиву
    print(frame.shape)
    print(frame)
    print('---')

##### Можлива структура вимірів зображень

![ARRAYS](pic/img_arr_structure.png)

# Зображення як масиви NumPy
---

### Зчитування та відображення зображень

In [ ]:
# demo_image = skimage.data.human_mitosis()
image = tiff.imread('demo_data/2D_grey_matter_neurofilaments.tif')
print(image.dtype)
print(image.shape)

In [ ]:
plt.figure(figsize=(5,5))
plt.imshow(image, cmap='Greys_r')

In [ ]:
crop_image = image[1100:1500,350:750]
print(crop_image.shape)

plt.figure(figsize=(5,5))
plt.imshow(crop_image, cmap='Greys_r')

### Кольорові мапи

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

def plot_linearmap(cdict):
    ''' Функція для візуалізації кольорової мапи.
    Не питайте, як це працює, це matplotlib-магія з глибин stack-overflow.
    
    '''
    newcmp = LinearSegmentedColormap('testCmap', segmentdata=cdict, N=256)
    rgba = newcmp(np.linspace(0, 1, 256))
    fig, ax = plt.subplots(figsize=(4, 3), constrained_layout=True)
    col = ['r', 'g', 'b']
    for i in range(3):
        ax.plot(np.arange(256)/256, rgba[:, i], color=col[i])
    ax.set_xlabel('itensity')
    ax.set_ylabel('RGB')
    plt.show()

Однокольорові мапи

In [ ]:
# створення та відображення кольорової мапи
dict_red = {'red':(
            (0.0, 0.0, 0.0),
            # (0.25, 0.1, 0.1),  # збільшення контрасту шляхом заниження яскравості пікселів низької інтенсивності
            (1.0, 1.0, 1.0)),
            'blue':(
            (0.0, 0.0, 0.0),
            (1.0, 0.0, 0.0)),
            'green':(
            (0.0, 0.0, 0.0),
            (1.0, 0.0, 0.0))}
cmap_red = LinearSegmentedColormap('Red_cmap', dict_red)

plt.figure(figsize=(5,5))
plt.imshow(crop_image, cmap=cmap_red)


plot_linearmap(dict_red)

Багатоколірні мапи

In [ ]:
dict_blue_red = {'red':(
                 (0.0, 0.0, 0.0),
                 (0.6, 0.25, 0.25),
                 (1.0, 1.0, 1.0)),
                 'blue':(
                 (0.0, 1.0, 1.0),
                 (0.4, 0.25, 0.25),
                 (1.0, 0.0, 0.0)),
                 'green':(
                 (0.0, 0.0, 0.0),
                 (0.2, 0.0, 0.0),
                 (0.5, 0.75, 0.75),
                 (0.8, 0.0, 0.0),
                 (1.0, 0.0, 0.0))}
cmap_blue_red = LinearSegmentedColormap('BR_cmap', dict_blue_red)

plt.figure(figsize=(5,5))
plt.imshow(crop_image, cmap=cmap_blue_red)

plot_linearmap(dict_blue_red)

# Корекція фонової інтенсивності
---

Основними методами корекції фонової інтенсивності є:
- Оцінка фону за знімком поля зору без зразка (dark frame)
- Оцінка фону як середнього значення інтенсивності пікселів вільної ділянки фону на зображенні 
- Оцінка фону за гістограмою
- Оцінка методом rolling ball

### Гістограма зображення

In [ ]:
print(crop_image.min())
print(crop_image.max())

plt.figure(figsize=(15,5))
plt.hist(crop_image.ravel(), bins=256)
plt.show()

Візьмемо більш драматичний випадок для ілюстративності

In [ ]:
bad_image = tiff.imread('demo_data/4D_HEK_spectral_time_series.tiff')[1,4]
print(bad_image.shape)
print(bad_image.dtype)
print(bad_image.min())
print(bad_image.max())

plt.figure(figsize=(5,5))
plt.imshow(bad_image, cmap='jet')

In [ ]:
plt.figure(figsize=(15,5))
plt.hist(bad_image.ravel(), bins=256)
plt.show()

In [ ]:
plt.figure(figsize=(15,5))
plt.hist(bad_image.ravel(), bins=256)
plt.yscale('log')
plt.show()

### Оцінка фонової інтенсивності за фрагментом зображення

In [ ]:
roi_background_int = np.mean(bad_image[0:50, 270:], dtype=np.uint32)
print(roi_background_int)

import matplotlib.patches as patches
crop_rect = patches.Rectangle((0, 270), 50, 50,
                              linewidth=1.5, edgecolor='white', facecolor='none')

fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(bad_image, cmap='jet')
ax.add_patch(crop_rect)
plt.show()

In [ ]:
bad_image_corrected = bad_image - roi_background_int

fig, (ax0, ax1) = plt.subplots(ncols=2, figsize=(10,5))

ax0.hist(bad_image.ravel(), bins=256)
# ax0.set_yscale('log')
ax0.set_title('Raw')

ax1.hist(bad_image_corrected.ravel(), bins=256)
# ax1.set_yscale('log')
ax1.set_title('Without background')

plt.show()

In [ ]:
plt.subplots(figsize=(5,5))
plt.imshow(bad_image_corrected, cmap='jet')
plt.show()

In [ ]:
# у випадку використання цілих чисел відбувається переповнення
a = np.uint16(150)
b = np.uint16(160)
c = a - b
print(c)

In [ ]:
# а використання чисел із плаваючою комою розмір зображення збільшиться вдвічі
a = np.float32(150)
b = np.uint16(160)
c = a - b
print(c)
print(c.dtype)

### Оцінка фонової інтенсивності за гістограмою

In [ ]:
# перший перцентіль значень інтенсивності як оцінка фону є більш стіким до коливань фону на відміну від мінімального значення (0 перцентиль)
perc_background_int = np.percentile(bad_image, 1)
perc_background_int

In [ ]:
# віднімання фону і перевірка на наявність від'ємних значень
bad_image_pre_corrected = bad_image - perc_background_int
bad_image_pre_corrected.min()

In [ ]:
# корекція від'ємних значень
bad_image_corrected = bad_image_pre_corrected.clip(min=0)
bad_image_corrected.min()

In [ ]:
# конвертація відкорегованого зображення
print(f'Data type after correction: {bad_image_corrected.dtype}, image size {bad_image_corrected.nbytes/1000} kB')

bad_image_preprocessed = bad_image_corrected.astype(np.uint16)

print(f'Final data type: {bad_image_preprocessed.dtype}, image size {bad_image_preprocessed.nbytes/1000} kB')

plt.subplots(figsize=(5,5))
plt.imshow(bad_image_preprocessed, cmap='jet')
plt.show()

### Оформлення кроків корекції фонової інтенсивності у функції

Оформлення всіх кроків корекції фонової інтенсивності у функцію

> [!NOTE]
> Раджу збирати функції з ноутбуків в модулі, щоб не копіювати код з ноутбука в ноутбук і оформити в пакет в подальшому
>

In [ ]:
def background_correction(input_img:np.ndarray, background_percentile:float=1):
    back_int = np.percentile(input_img, background_percentile)
    corr_img = input_img - back_int
    corr_img = corr_img.clip(min=0)
    return corr_img.astype(input_img.dtype)

Альтернативний варіант оформлення функції з використанням __list comprehension__ та __lambda-функції__

In [ ]:
bc = lambda img, p=1.0:np.array(img - np.percentile(img, p)).clip(min=0).astype(dtype=img.dtype)

### Збереження зображення після попередньої обробки

In [ ]:
tiff.imwrite('course_data/preproc_img.tif', background_correction(bad_image))

### Корекція фонової інтенсивності із використанням rolling-ball

Для детального опису варто звернутись до [документації scikit-image](https://scikit-image.org/docs/0.25.x/auto_examples/segmentation/plot_rolling_ball.html)

In [ ]:
background_image = skimage.restoration.rolling_ball(bad_image, radius=200)
print(f'Background image data type: {background_image.dtype}')

bad_image_rb = bad_image - background_image
print(f'Corrected image data type: {bad_image_rb.dtype}')


plt.figure(figsize=(12,6))

ax0 = plt.subplot(121)
ax0.imshow(bad_image)
ax0.set_title('Raw image')

ax1 = plt.subplot(122)
ax1.imshow(background_image)
ax1.set_title('Background')

plt.show()

Багатопанельні зображення з використанням matplotlib

In [ ]:
plt.figure(figsize=(12,6))

# зображення
ax0 = plt.subplot(231)
ax0.imshow(bad_image)
ax0.set_title('Raw image')
ax0.axis('off')

ax1 = plt.subplot(232)
ax1.imshow(background_image)
ax1.set_title('Background')
ax1.axis('off')

ax2 = plt.subplot(233)
ax2.imshow(bad_image_rb)
ax2.set_title('Corrected image')
ax2.axis('off')

# гістограми
ax3 = plt.subplot(234)
ax3.hist(bad_image.ravel(), bins=256)
ax3.set_title('Raw image')

ax4 = plt.subplot(235)
ax4.hist(background_image.ravel(), bins=256)
ax4.set_title('Background')

ax5 = plt.subplot(236)
ax5.hist(bad_image_rb.ravel(), bins=256)
ax5.set_title('Corrected image')

plt.suptitle('Background estimation with rolling-ball')
plt.tight_layout()
plt.show()

# Завдання
---

- __За відсутності власних зображень:__ обрати із демонстраційних зображення для подальшої роботи та ознайомитись із завданнями щодо його аналізу.
- __За наявності власних зображень:__ сформувати перелік завдань щодо аналізу власних даних, спробувати сформулювати характеристики чи величини, що необхідно оцінити і які кроки передують отриманню значень цих характеристик/величин.
- Провести попередню обробку обраних для подальшої роботи даних, за необхідності провести кроп зображення щоб лишити лише значущі регіони і провести корекцію фонової інтенсивності.
- _Опціонально_ адапувати одну з наведених вище функції для корекції фонової інтенсивності для використання із 3D-зображеннями та часовими серіями зображень.
- _Опіціонально_ створити функцію для корекції фонової інтенсивності, що приймала би на всіх початкове зображення та координати регіону, ща середньою інтенсивністю якого відбувається оцінка фонової інтенсивності (координати можна передавати функції у вигляді списку виду `[x_start, x_end, y_start, y_end]`).